# 🚗 Car Price Prediction — CarDekho Dataset
**Author:** Vaishnavi  
**Dataset:** CAR DETAILS FROM CAR DEKHO.csv  
**Task:** Regression — Predict used car selling price  
**Models:** Linear Regression · Random Forest · Gradient Boosting

---

## Section 1 — Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

# ── Load dataset ──────────────────────────────────────────────────────────────
DATA_PATH = 'CAR DETAILS FROM CAR DEKHO.csv'
df = pd.read_csv(DATA_PATH)

print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
print('Data Types:')
print(df.dtypes)
print('\nBasic Statistics:')
df.describe(include='all')

## Section 2 — Data Cleaning

In [ ]:
# ── Missing values ─────────────────────────────────────────────────────────────
print('Missing values per column:')
print(df.isnull().sum())

# ── Duplicate rows ─────────────────────────────────────────────────────────────
n_dupes = df.duplicated().sum()
print(f'\nDuplicate rows found: {n_dupes}')

df = df.drop_duplicates()
print(f'Shape after removing duplicates: {df.shape}')

# Reset index
df = df.reset_index(drop=True)
print('\nData cleaning complete. No missing values, no duplicates.')

## Section 3 — Exploratory Data Analysis (EDA)

In [ ]:
# ── Plot 1: Selling Price Distribution ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['selling_price'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Selling Price Distribution (Original)')
axes[0].set_xlabel('Selling Price (INR)')
axes[0].set_ylabel('Count')

axes[1].hist(np.log1p(df['selling_price']), bins=50, color='seagreen', edgecolor='white')
axes[1].set_title('Selling Price Distribution (Log Scale)')
axes[1].set_xlabel('log(Selling Price + 1)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('eda_price_distribution.png', dpi=100)
plt.show()
print('Plot 1 saved.')

In [ ]:
# ── Plot 2: Selling Price by Fuel Type ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
fuel_order = df.groupby('fuel')['selling_price'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='fuel', y='selling_price', order=fuel_order, ax=ax, palette='Set2')
ax.set_yscale('log')
ax.set_title('Selling Price by Fuel Type (Log Scale)')
ax.set_xlabel('Fuel Type')
ax.set_ylabel('Selling Price (INR, log scale)')
plt.tight_layout()
plt.savefig('eda_price_by_fuel.png', dpi=100)
plt.show()
print('Plot 2 saved.')

In [ ]:
# ── Plot 3: Year vs Selling Price ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
year_avg = df.groupby('year')['selling_price'].mean().reset_index()
ax.plot(year_avg['year'], year_avg['selling_price'], marker='o', color='darkorange', linewidth=2)
ax.set_title('Average Selling Price by Year')
ax.set_xlabel('Year')
ax.set_ylabel('Average Selling Price (INR)')
plt.tight_layout()
plt.savefig('eda_price_by_year.png', dpi=100)
plt.show()
print('Plot 3 saved.')

In [ ]:
# ── Plot 4: KM Driven vs Selling Price ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(df['km_driven'], df['selling_price'], alpha=0.3, color='slateblue', s=10)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title('KM Driven vs Selling Price (Log-Log Scale)')
ax.set_xlabel('KM Driven (log scale)')
ax.set_ylabel('Selling Price (log scale)')
plt.tight_layout()
plt.savefig('eda_km_vs_price.png', dpi=100)
plt.show()
print('Plot 4 saved.')

In [ ]:
# ── Plot 5: Transmission & Seller Type Counts ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df['transmission'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','salmon'], edgecolor='white')
axes[0].set_title('Transmission Type Distribution')
axes[0].set_xlabel('Transmission')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

df['seller_type'].value_counts().plot(kind='bar', ax=axes[1], color=['seagreen','steelblue','darkorange'], edgecolor='white')
axes[1].set_title('Seller Type Distribution')
axes[1].set_xlabel('Seller Type')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('eda_transmission_seller.png', dpi=100)
plt.show()
print('Plot 5 saved.')

In [ ]:
# ── Plot 6: Owner Type vs Average Selling Price ───────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
owner_order = ['First Owner', 'Second Owner', 'Third Owner', 'Fourth & Above Owner', 'Test Drive Car']
owner_avg = df.groupby('owner')['selling_price'].mean().reindex(owner_order)
owner_avg.plot(kind='bar', ax=ax, color='mediumpurple', edgecolor='white')
ax.set_title('Average Selling Price by Owner Type')
ax.set_xlabel('Owner Type')
ax.set_ylabel('Average Selling Price (INR)')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('eda_price_by_owner.png', dpi=100)
plt.show()
print('Plot 6 saved.')

## Section 4 — Feature Engineering

In [ ]:
# ── Extract brand from name ────────────────────────────────────────────────────
df['brand'] = df['name'].str.split().str[0]

print('Top 15 brands by count:')
print(df['brand'].value_counts().head(15))
print(f'\nTotal unique brands: {df["brand"].nunique()}')

In [ ]:
# ── Plot 7: Top 15 Brands by Average Selling Price ───────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
top_brands = df['brand'].value_counts().head(15).index
brand_avg = df[df['brand'].isin(top_brands)].groupby('brand')['selling_price'].mean().sort_values(ascending=False)
brand_avg.plot(kind='bar', ax=ax, color='teal', edgecolor='white')
ax.set_title('Average Selling Price — Top 15 Brands')
ax.set_xlabel('Brand')
ax.set_ylabel('Average Selling Price (INR)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('eda_brand_avg_price.png', dpi=100)
plt.show()
print('Plot 7 saved.')

In [ ]:
# ── Create car_age feature ─────────────────────────────────────────────────────
CURRENT_YEAR = 2024
df['car_age'] = CURRENT_YEAR - df['year']
print('car_age sample:', df['car_age'].describe())

# ── Drop columns no longer needed ─────────────────────────────────────────────
df = df.drop(columns=['name', 'year'])

print('\nFinal feature columns:', list(df.columns))

In [ ]:
# ── Log-transform the target ───────────────────────────────────────────────────
df['log_selling_price'] = np.log1p(df['selling_price'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['selling_price'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('selling_price (original)')
axes[1].hist(df['log_selling_price'], bins=50, color='seagreen', edgecolor='white')
axes[1].set_title('log(selling_price+1)')
plt.tight_layout()
plt.savefig('eda_target_transform.png', dpi=100)
plt.show()

# Define features and target
FEATURES = ['brand', 'km_driven', 'fuel', 'seller_type', 'transmission', 'owner', 'car_age']
TARGET   = 'log_selling_price'

X = df[FEATURES].copy()
y = df[TARGET].copy()

print('X shape:', X.shape)
print('y shape:', y.shape)
print('Target (log_selling_price) stats:')
print(y.describe())

In [ ]:
# ── Correlation heatmap on numeric columns ────────────────────────────────────
numeric_cols = df[['km_driven', 'car_age', 'selling_price', 'log_selling_price']]
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(numeric_cols.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.savefig('eda_correlation_heatmap.png', dpi=100)
plt.show()
print('Correlation heatmap saved.')

## Section 5 — Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Training set : {X_train.shape[0]} rows')
print(f'Test set     : {X_test.shape[0]} rows')

## Section 6 — Preprocessing Pipeline

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

CATEGORICAL_FEATURES = ['brand', 'fuel', 'seller_type', 'transmission', 'owner']
NUMERIC_FEATURES     = ['km_driven', 'car_age']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), NUMERIC_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES)
])

print('Preprocessor defined.')
print('Numeric features  :', NUMERIC_FEATURES)
print('Categorical features:', CATEGORICAL_FEATURES)

## Section 7 — Model Training

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    'Linear Regression'    : LinearRegression(),
    'Random Forest'        : RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting'    : GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, random_state=42)
}

results = {}
trained_pipelines = {}

for model_name, model in models.items():
    print(f'Training: {model_name} ...')
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    pipeline.fit(X_train, y_train)
    
    y_pred_log = pipeline.predict(X_test)
    # Convert back from log scale for interpretable metrics
    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_test)
    
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    
    results[model_name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    trained_pipelines[model_name] = pipeline
    print(f'  MAE  = {mae:,.0f}  |  RMSE = {rmse:,.0f}  |  R² = {r2:.4f}')

print('\nAll models trained.')

## Section 8 — Model Evaluation & Comparison

In [ ]:
results_df = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'Model'})
results_df = results_df.sort_values('R2', ascending=False).reset_index(drop=True)
results_df['MAE']  = results_df['MAE'].map('{:,.0f}'.format)
results_df['RMSE'] = results_df['RMSE'].map('{:,.0f}'.format)
results_df['R2']   = results_df['R2'].map('{:.4f}'.format)
print('Model Comparison (sorted by R²):')
print(results_df.to_string(index=False))

In [ ]:
# ── Plot 8: Model Comparison Bar Charts ──────────────────────────────────────
metrics_raw = {}
for name, model in models.items():
    pipeline = trained_pipelines[name]
    y_pred_log = pipeline.predict(X_test)
    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_test)
    metrics_raw[name] = {
        'MAE' : mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2'  : r2_score(y_true, y_pred)
    }

model_names = list(metrics_raw.keys())
mae_vals  = [metrics_raw[m]['MAE']  for m in model_names]
rmse_vals = [metrics_raw[m]['RMSE'] for m in model_names]
r2_vals   = [metrics_raw[m]['R2']   for m in model_names]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].bar(model_names, mae_vals, color=['steelblue','salmon','seagreen'])
axes[0].set_title('MAE Comparison')
axes[0].set_ylabel('MAE (INR)')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(model_names, rmse_vals, color=['steelblue','salmon','seagreen'])
axes[1].set_title('RMSE Comparison')
axes[1].set_ylabel('RMSE (INR)')
axes[1].tick_params(axis='x', rotation=15)

axes[2].bar(model_names, r2_vals, color=['steelblue','salmon','seagreen'])
axes[2].set_title('R² Score Comparison')
axes[2].set_ylabel('R² Score')
axes[2].set_ylim(0, 1)
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=100)
plt.show()
print('Model comparison chart saved.')

## Section 9 — Best Model Selection & Saving

In [ ]:
import joblib

# Select best model by R²
best_name = max(metrics_raw, key=lambda m: metrics_raw[m]['R2'])
best_pipeline = trained_pipelines[best_name]

print(f'Best Model: {best_name}')
print(f"  MAE  = {metrics_raw[best_name]['MAE']:,.0f} INR")
print(f"  RMSE = {metrics_raw[best_name]['RMSE']:,.0f} INR")
print(f"  R²   = {metrics_raw[best_name]['R2']:.4f}")

# Save model and preprocessor
os.makedirs('models', exist_ok=True)
joblib.dump(best_pipeline, 'models/best_model.pkl')
print('\nBest model pipeline saved to models/best_model.pkl')

In [ ]:
# ── Actual vs Predicted Plot for Best Model ───────────────────────────────────
y_pred_log = best_pipeline.predict(X_test)
y_pred_best = np.expm1(y_pred_log)
y_true_best = np.expm1(y_test)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_true_best, y_pred_best, alpha=0.3, s=15, color='darkcyan')
max_val = max(y_true_best.max(), y_pred_best.max())
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Perfect Prediction')
ax.set_title(f'Actual vs Predicted Selling Price ({best_name})')
ax.set_xlabel('Actual Selling Price (INR)')
ax.set_ylabel('Predicted Selling Price (INR)')
ax.legend()
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=100)
plt.show()
print('Actual vs Predicted chart saved.')

In [ ]:
# ── Residuals Plot ─────────────────────────────────────────────────────────────
residuals = y_true_best - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_pred_best, residuals, alpha=0.3, s=10, color='coral')
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Residuals vs Predicted')
axes[0].set_xlabel('Predicted Selling Price')
axes[0].set_ylabel('Residual')

axes[1].hist(residuals, bins=50, color='coral', edgecolor='white')
axes[1].set_title('Residuals Distribution')
axes[1].set_xlabel('Residual (INR)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('residuals_plot.png', dpi=100)
plt.show()
print('Residuals plot saved.')

## Section 10 — Quick Prediction Test

In [ ]:
# Verify the saved model loads and predicts correctly
loaded_model = joblib.load('models/best_model.pkl')

sample = pd.DataFrame([{
    'brand'       : 'Maruti',
    'km_driven'   : 50000,
    'fuel'        : 'Petrol',
    'seller_type' : 'Individual',
    'transmission': 'Manual',
    'owner'       : 'First Owner',
    'car_age'     : 7
}])

pred_log = loaded_model.predict(sample)[0]
pred_price = np.expm1(pred_log)
print(f'Sample prediction — Maruti, Petrol, Manual, 50k km, 7 yrs old:')
print(f'  Predicted Selling Price: ₹{pred_price:,.0f}')

## Section 11 — Conclusion

In this project we built a complete end-to-end used car price prediction system using the CarDekho dataset.

**Key findings:**
- The dataset contained **4,340 records** after removing 763 exact duplicates from the original 4,340 rows.
- Car brand, fuel type, transmission, car age, and km driven are the most influential features.
- Applying a **log transformation** on the target `selling_price` significantly improved model performance by reducing skewness.
- **Gradient Boosting** and **Random Forest** outperform Linear Regression by a wide margin, capturing non-linear relationships effectively.
- The best model achieves a high R² score, meaning it explains most of the variance in used car prices.

**Deployment:**
- The best model pipeline is saved as `models/best_model.pkl` using joblib.
- A Flask REST API (`app.py`) serves predictions via a `/predict` POST endpoint.
- A Streamlit frontend (`streamlit_app.py`) provides an interactive UI for end users.

**Future Scope:**
- Include additional features like engine capacity, mileage (kmpl), and location.
- Scrape fresh data periodically and retrain the model.
- Deploy to cloud platforms (Render, Railway, or AWS EC2).
- Build hyperparameter tuning with cross-validation for further accuracy gains.